### Step 1: Ingestion & Environment Setup
- **Objective:** Mount Google Drive, extract the dataset zip file, and verify dataset features.

In [57]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
!unzip "/content/drive/MyDrive/data_windowed.zip" -d "/content/dataset"

Archive:  /content/drive/MyDrive/data_windowed.zip
replace /content/dataset/data_windowed.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/dataset/data_windowed.csv  A



In [59]:
import os

files = os.listdir('/content/dataset')
print("Extracted files:", files)

Extracted files: ['data_windowed.csv']


### Step 2: Data Loading & Timestamp Parsing
- **Objective:** Load the time-series dataset, parse ISO-8601 timestamps, and establish the overall timeline boundaries.

In [60]:
import pandas as pd

csv_path = f'/content/dataset/{files[0]}'

df_sample = pd.read_csv(csv_path, nrows=5)
print("Columns count:", len(df_sample.columns))
print("Columns:", df_sample.columns.tolist())

Columns count: 19
Columns: ['timestamp', 'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses', 'segment', 'ground_truth', 'window_id']


In [61]:
data_path = '/content/dataset/data_windowed.csv'
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(len(df))
print(df['timestamp'].min())
print(df['timestamp'].max())

1516948
2020-02-01 00:00:00
2020-09-01 03:59:50


### Step 3: Feature & Metadata Separation
- **Objective:** Separate raw sensor measurement columns (`X_features`) from metadata columns (`timestamp`, `segment`, `ground_truth`, `window_id`).

In [62]:
excluded_cols = ['timestamp', 'segment', 'ground_truth', 'window_id']
sensor_cols = [col for col in df.columns if col not in excluded_cols]

X_features = df[sensor_cols].copy()
timestamp_index = df[excluded_cols].copy()

print(X_features.shape)
print(timestamp_index.shape)

(1516948, 15)
(1516948, 4)


In [63]:
import numpy as np

cutoff_train_val = '2020-06-01 00:00:00'
cutoff_val_test = '2020-07-01 00:00:00'

conditions = [
    timestamp_index['timestamp'] < cutoff_train_val,
    (timestamp_index['timestamp'] >= cutoff_train_val)
    & (timestamp_index['timestamp'] < cutoff_val_test),
    timestamp_index['timestamp'] >= cutoff_val_test,
]

splits = ['train', 'validation', 'test']
timestamp_index['split'] = np.select(conditions, splits, default='test')

print(timestamp_index['split'].value_counts())

split
train         856832
test          443602
validation    216514
Name: count, dtype: int64


In [64]:
train_max = timestamp_index[timestamp_index['split'] == 'train'][
    'timestamp'
].max()
val_min = timestamp_index[timestamp_index['split'] == 'validation'][
    'timestamp'
].min()
val_max = timestamp_index[timestamp_index['split'] == 'validation'][
    'timestamp'
].max()
test_min = timestamp_index[timestamp_index['split'] == 'test'][
    'timestamp'
].min()

assert train_max < val_min < val_max < test_min
assert len(X_features) == len(df)
assert len(timestamp_index) == len(df)
assert X_features.isna().sum().sum() == 0

print("All assertions passed successfully.")

All assertions passed successfully.


In [65]:
import json
import os

output_dir = '/content/phase2_task1_outputs'
os.makedirs(output_dir, exist_ok=True)

X_features.to_csv(f'{output_dir}/X_features.csv', index=False)
timestamp_index.to_csv(f'{output_dir}/timestamp_index.csv', index=False)

config = {
    'ordered_features': sensor_cols,
    'excluded_columns': excluded_cols,
    'cutoff_timestamps': {
        'train_val': cutoff_train_val,
        'val_test': cutoff_val_test,
    },
    'failures_distribution': {
        'train': [1, 2],
        'validation': [3],
        'test': [4],
    },
    'total_rows': len(df),
    'splits_count': timestamp_index['split'].value_counts().to_dict(),
}

with open(f'{output_dir}/features_config.json', 'w') as f:
  json.dump(config, f, indent=4)

print("Files saved successfully.")

Files saved successfully.


In [66]:
report_content =rf"""# Phase 2 - Task 1: Feature Separation & Event-Based Split Report

## 1. Overview & Data Summary
- **Total Samples:** {len(df):,}
- **Time Range:** {df['timestamp'].min()} to {df['timestamp'].max()}
- **Sensor Features Count:** {X_features.shape[1]}
- **Metadata Columns:** {excluded_cols}

---

## 2. Cutoff Timestamps & Split Logic
Splits were constructed based on historical failure events to prevent temporal leakage and evaluate model performance across unseen failures:
- **Cutoff 1 (Train / Validation):** `{cutoff_train_val}`
- **Cutoff 2 (Validation / Test):** `{cutoff_val_test}`

---

## 3. Failure Events Distribution
- **Train Set:** Event 1 (2020-04-18) and Event 2 (2020-05-29 to 2020-05-30)
- **Validation Set:** Event 3 (2020-06-05 to 2020-06-07)
- **Test Set:** Event 4 (2020-07-15)

---

## 4. Split Statistics
| Split | Row Count | Percentage |
| :--- | :--- | :--- |
| **Train** | 856,832 | 56.48% |
| **Validation** | 216,514 | 14.27% |
| **Test** | 443,602 | 29.24% |
| **Total** | **1,516,948** | **100.00%** |

---

## 5. Verification & Acceptance Criteria
- **Temporal Order:** Verified ($\max(Train) < \min(Val) < \min(Test)$)[cite: 7].
- **Data Integrity:** No missing values in sensor features, row counts strictly match source dataset[cite: 7].
"""

with open(f"{output_dir}/split_report.md", "w") as f:
    f.write(report_content)

print("split_report.md created successfully.")

split_report.md created successfully.


In [67]:
import shutil
from google.colab import files

shutil.make_archive('phase2_task1_outputs', 'zip', '/content/phase2_task1_outputs')
files.download('phase2_task1_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [68]:
print(df.groupby('segment')['timestamp'].agg(['min', 'max', 'count']))

                        min                 max  count
segment                                               
0       2020-02-01 00:00:00 2020-02-01 12:48:40   4654
1       2020-02-01 12:53:47 2020-02-01 19:40:04   2460
2       2020-02-01 23:15:33 2020-02-01 23:20:21     30
3       2020-02-02 04:34:27 2020-02-02 07:37:40   1110
4       2020-02-02 07:41:52 2020-02-05 00:32:21  23550
...                     ...                 ...    ...
327     2020-08-30 04:26:10 2020-08-31 00:30:23   7290
328     2020-08-31 01:49:32 2020-08-31 06:36:50   1740
329     2020-08-31 06:42:31 2020-08-31 20:15:10   4920
330     2020-09-01 00:37:33 2020-09-01 00:42:20     30
331     2020-09-01 02:37:24 2020-09-01 03:59:50    500

[332 rows x 3 columns]


In [69]:
sensor_cols = [c for c in df.columns if c not in ['timestamp', 'segment', 'ground_truth', 'window_id']]
results = []

for col in sensor_cols:
    is_col_frozen = (df[col].diff() == 0)
    groups = (~is_col_frozen).cumsum()
    counts = is_col_frozen.groupby(groups).sum()
    if len(counts) > 0 and counts.max() > 100:
        max_id = counts.idxmax()
        group_df = df[groups == max_id]
        results.append({
            'sensor': col,
            'frozen_rows': int(counts.max()),
            'start': group_df['timestamp'].min(),
            'end': group_df['timestamp'].max()
        })

res_df = pd.DataFrame(results)
if not res_df.empty:
    print(res_df.sort_values(by='frozen_rows', ascending=False))
else:
    print("No frozen sensors found.")

             sensor  frozen_rows               start                 end
14  Caudal_impulses       287972 2020-07-22 13:04:51 2020-09-01 03:59:50
11              LPS       254010 2020-02-06 07:52:30 2020-03-11 19:21:46
13        Oil_level       233668 2020-03-12 14:11:12 2020-04-13 18:31:09
3       DV_pressure        18540 2020-06-22 15:01:53 2020-06-25 05:08:35
7              COMP        18517 2020-06-22 15:05:41 2020-06-25 05:08:35
10              MPG        18517 2020-06-22 15:05:41 2020-06-25 05:08:35
1               TP3        18514 2020-06-22 15:06:11 2020-06-25 05:08:35
6     Motor_current        18514 2020-06-22 15:06:11 2020-06-25 05:08:35
5   Oil_temperature        18514 2020-06-22 15:06:11 2020-06-25 05:08:35
4        Reservoirs        18514 2020-06-22 15:06:11 2020-06-25 05:08:35
2                H1        18514 2020-06-22 15:06:11 2020-06-25 05:08:35
0               TP2        18514 2020-06-22 15:06:11 2020-06-25 05:08:35
8        DV_eletric        17950 2020-06-05 09:48:3

### Step 4: Frozen Period Audit & Removal
- **Objective:** Detect and remove periods where sensor readings remained static/frozen for prolonged durations to ensure high data integrity.

In [70]:
frozen_start = '2020-06-22 15:05:00'
frozen_end = '2020-06-25 05:08:35'

mask_frozen = (df['timestamp'] >= frozen_start) & (df['timestamp'] <= frozen_end)
df = df[~mask_frozen].copy()

print(len(df))

1498426


In [71]:
excluded_cols = ['timestamp', 'segment', 'ground_truth', 'window_id']
sensor_cols = [col for col in df.columns if col not in excluded_cols]

X_features = df[sensor_cols].copy()
timestamp_index = df[excluded_cols].copy()

print(X_features.shape)
print(timestamp_index.shape)

(1498426, 15)
(1498426, 4)


### Step 5: Event-Based Temporal Splitting
- **Objective:** Split data into train, validation, and test sets based on failure event cutoff timestamps, ensuring zero failure leakage in the training set (`ground_truth = 0` only).

In [72]:
import numpy as np

cutoff_train_val = '2020-04-12 00:00:00'
cutoff_val_test = '2020-07-01 00:00:00'

conditions = [
    timestamp_index['timestamp'] < cutoff_train_val,
    (timestamp_index['timestamp'] >= cutoff_train_val) & (timestamp_index['timestamp'] < cutoff_val_test),
    timestamp_index['timestamp'] >= cutoff_val_test
]

splits = ['train', 'validation', 'test']
timestamp_index['split'] = np.select(conditions, splits, default='test')

train_idx = timestamp_index['split'] == 'train'
leakage_mask = train_idx & (timestamp_index['ground_truth'] != 0)
timestamp_index.loc[leakage_mask, 'split'] = 'validation'

print(timestamp_index['split'].value_counts())

split
validation    531976
train         522848
test          443602
Name: count, dtype: int64


### Step 6: Verification & Self-Checks
- **Objective:** Run programmatic assertions to verify temporal ordering, check data integrity, and ensure zero missing values.

In [73]:
train_max = timestamp_index[timestamp_index['split'] == 'train']['timestamp'].max()
val_min = timestamp_index[timestamp_index['split'] == 'validation']['timestamp'].min()
val_max = timestamp_index[timestamp_index['split'] == 'validation']['timestamp'].max()
test_min = timestamp_index[timestamp_index['split'] == 'test']['timestamp'].min()

assert train_max < val_min < val_max < test_min
assert len(X_features) == len(df)
assert len(timestamp_index) == len(df)
assert X_features.isna().sum().sum() == 0

print("All assertions passed successfully.")

All assertions passed successfully.


### Step 7: Exporting Outputs & Configuration
- **Objective:** Save processed features (`X_features.csv`), metadata indices (`timestamp_index.csv`), update configuration (`features_config.json` with `random_state = 42`), and generate the final split report.

In [74]:
import json
import os

output_dir = '/content/phase2_task1_outputs'
os.makedirs(output_dir, exist_ok=True)

X_features.to_csv(f'{output_dir}/X_features.csv', index=False)
timestamp_index.to_csv(f'{output_dir}/timestamp_index.csv', index=False)

train_count = int((timestamp_index['split'] == 'train').sum())
val_count = int((timestamp_index['split'] == 'validation').sum())
test_count = int((timestamp_index['split'] == 'test').sum())
total_count = len(df)

config = {
    "ordered_features": sensor_cols,
    "excluded_columns": excluded_cols,
    "frozen_period_removed": {
        "start": frozen_start,
        "end": frozen_end,
        "rows_removed": int(mask_frozen.sum())
    },
    "cutoff_timestamps": {
        "train_val": cutoff_train_val,
        "val_test": cutoff_val_test
    },
    "failures_distribution": {
        "train": [1, 2],
        "validation": [3],
        "test": [4]
    },
    "total_rows": total_count,
    "splits_count": {
        "train": train_count,
        "validation": val_count,
        "test": test_count
    }
}

with open(f'{output_dir}/features_config.json', 'w') as f:
    json.dump(config, f, indent=4)

report_content = rf"""# Phase 2 - Task 1: Feature Separation & Event-Based Split Report

## 1. Overview & Data Summary
- **Total Samples (After Cleaning):** {total_count:,}
- **Time Range:** {df['timestamp'].min()} to {df['timestamp'].max()}
- **Sensor Features Count:** {X_features.shape[1]}
- **Metadata Columns:** {excluded_cols}
- **Removed Frozen Period:** `{frozen_start}` to `{frozen_end}` ({int(mask_frozen.sum()):,} rows)

---

## 2. Cutoff Timestamps & Split Logic
Splits were constructed based on historical failure events:
- **Cutoff 1 (Train / Validation):** `{cutoff_train_val}`
- **Cutoff 2 (Validation / Test):** `{cutoff_val_test}`

---

## 3. Failure Events Distribution
- **Train Set:** Event 1 (2020-04-18) and Event 2 (2020-05-29 to 2020-05-30)
- **Validation Set:** Event 3 (2020-06-05 to 2020-06-07)
- **Test Set:** Event 4 (2020-07-15)

---

## 4. Split Statistics
| Split | Row Count | Percentage |
| :--- | :--- | :--- |
| **Train** | {train_count:,} | {train_count / total_count * 100:.2f}% |
| **Validation** | {val_count:,} | {val_count / total_count * 100:.2f}% |
| **Test** | {test_count:,} | {test_count / total_count * 100:.2f}% |
| **Total** | **{total_count:,}** | **100.00%** |

---

## 5. Verification & Acceptance Criteria
- **Temporal Order:** Verified ($\max(Train) < \min(Val) < \min(Test)$).
- **Data Integrity:** No missing values in sensor features, row counts match cleaned dataset.
"""

with open(f"{output_dir}/split_report.md", "w") as f:
    f.write(report_content)

print("Files saved successfully.")

Files saved successfully.


In [75]:
import shutil
from google.colab import files

shutil.make_archive('phase2_task1_cleaned_outputs', 'zip', '/content/phase2_task1_outputs')
files.download('phase2_task1_cleaned_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [76]:
import pandas as pd

ts_df = pd.read_csv('/content/phase2_task1_outputs/timestamp_index.csv')
train_failures = ts_df[(ts_df['split'] == 'train') & (ts_df['ground_truth'] != 0)]

print(f"Number of leaked failure rows in train: {len(train_failures)}")
assert len(train_failures) == 0, "AssertionError: Training set contains failure rows!"
print("Self-check passed successfully!")

Number of leaked failure rows in train: 0
Self-check passed successfully!


In [77]:
train_max_date = ts_df[ts_df['split'] == 'train']['timestamp'].max()
test_min_date = ts_df[ts_df['split'] == 'test']['timestamp'].min()
assert train_max_date < test_min_date
print("Temporal split check passed.")

Temporal split check passed.


In [78]:
x_df = pd.read_csv('/content/phase2_task1_outputs/X_features.csv')
assert len(x_df) == len(ts_df)
assert x_df.isna().sum().sum() == 0
print("Data integrity check passed.")

Data integrity check passed.


In [79]:
import json

with open('/content/phase2_task1_outputs/features_config.json', 'r') as f:
    cfg = json.load(f)

cfg["random_state"] = 42

with open('/content/phase2_task1_outputs/features_config.json', 'w') as f:
    json.dump(cfg, f, indent=4)

print("Config updated with random_state = 42.")

Config updated with random_state = 42.
